# Data exploration: Olist orders

Historical EDA from the Brazilian E-Commerce Public Dataset by Olist. Download CSVs from the credited original dataset; raw records are intentionally not included in this public repository.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("data/raw")
orders = pd.read_csv(DATA / "olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
])
print("Orders, columns:", orders.shape)
print("Unique order IDs:", orders.order_id.nunique())
display(orders.head())
display(orders.isna().sum().to_frame("missing"))
display(orders.order_status.value_counts().to_frame("count"))

## Descriptive late-delivery definition

For confirmed deliveries with valid dates, a delivery is late if its actual delivery *calendar date* exceeds its promised date. The observed outcome is not a purchase-time model feature.

In [ ]:
delivered = orders.loc[
    orders.order_status.eq("delivered")
    & orders.order_delivered_customer_date.notna()
    & orders.order_estimated_delivery_date.notna()
].copy()
delivered["late_delivery"] = (
    delivered.order_delivered_customer_date.dt.normalize()
    > delivered.order_estimated_delivery_date.dt.normalize()
).astype(int)
print("Confirmed delivered observations:", len(delivered))
print("Late delivery rate:", round(delivered.late_delivery.mean() * 100, 2), "%")

## Historical trend and cohort coverage

Some months have only a few records. Descriptive delivery rates are conditional on a recorded confirmed delivery, and should not be interpreted as non-delivery risk among all purchases.

In [ ]:
orders["purchase_month"] = orders.order_purchase_timestamp.dt.to_period("M")
delivered["purchase_month"] = delivered.order_purchase_timestamp.dt.to_period("M")
monthly = orders.groupby("purchase_month").size().to_frame("all_orders")
monthly = monthly.join(
    delivered.groupby("purchase_month").agg(
        confirmed_deliveries=("order_id", "size"),
        late_deliveries=("late_delivery", "sum"),
        late_rate=("late_delivery", "mean")
    )
)
monthly["coverage"] = monthly.confirmed_deliveries / monthly.all_orders
display(monthly.tail(18).round(3))
monthly["late_rate"].mul(100).plot(figsize=(11, 4), marker="o", title="Monthly late delivery rate among confirmed deliveries")
plt.ylabel("Late deliveries (%)")
plt.tight_layout()
plt.show()

## Additional exploration

The reusable aggregation, temporal cutoff, and model pipelines are implemented in src/train.py. For reproducibility, do not fit imputers or scalers on validation or test data.